# Demo — Converse Under the Hood

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/bedrock-companion/blob/main/day1/notebook-demos/demo-converse-under-the-hood/demo-converse-under-the-hood.ipynb)

This notebook makes live Amazon Bedrock calls. Students observe the normalized Converse response, token usage, stateless message handling, and a complete client-side tool-use exchange.

## Before you run

- Prefer running locally after `source ~/team-XX.env`; never paste AWS credentials into a notebook cell, output, screenshot, or committed file.
- If Colab is instructor-approved, inject credentials through its Secrets facility and remove them when the session ends.
- The credential must allow `sts:GetCallerIdentity` and `bedrock:InvokeModel`.
- The demo installs the course-pinned `boto3==1.43.62`, defaults to `amazon.nova-lite-v1:0` in `us-east-1`, and accepts instructor-provided `MODEL_ID` and `AWS_REGION` overrides.
- The local case lookup is intentionally deterministic; Amazon Bedrock performs the reasoning and tool selection, while the application executes the function.

In [1]:
%pip install -q "boto3==1.43.62"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 94.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 7.9 MB/s eta 0:00:00


In [2]:
import json
import os
import time

import boto3

AWS_REGION = os.environ.get("AWS_REGION", os.environ.get("AWS_DEFAULT_REGION", "us-east-1"))
MODEL_ID = os.environ.get("MODEL_ID", "amazon.nova-lite-v1:0")
CASE_NUMBER = "00001042"
CASE_RECORDS = {
    CASE_NUMBER: {
        "caseNumber": CASE_NUMBER,
        "priority": "HIGH",
        "status": "ESCALATED",
        "subject": "Partner portal access",
    }
}
SYSTEM_PROMPT = (
    "You are a concise Salesforce support assistant. "
    "If a requested case number is absent from the messages, answer exactly NO_CASE_CONTEXT. "
    "When a case status is requested and the lookup_case tool is available, use the tool instead of guessing."
)
TOOL_CONFIG = {
    "tools": [
        {
            "toolSpec": {
                "name": "lookup_case",
                "description": "Read the current status and priority of a synthetic Salesforce support case.",
                "inputSchema": {
                    "json": {
                        "type": "object",
                        "properties": {
                            "caseNumber": {
                                "type": "string",
                                "description": "The eight-digit Salesforce case number.",
                            }
                        },
                        "required": ["caseNumber"],
                    }
                },
            }
        }
    ]
}
FORCED_TOOL_CONFIG = {
    "tools": TOOL_CONFIG["tools"],
    "toolChoice": {"tool": {"name": "lookup_case"}},
}

bedrock = boto3.client("bedrock-runtime", region_name=AWS_REGION)
sts = boto3.client("sts", region_name=AWS_REGION)


def print_section(title):
    print(f"\n{'=' * 72}\n{title}\n{'=' * 72}")


def message_text(message):
    return "".join(block.get("text", "") for block in message.get("content", []))


def show_response(label, response, elapsed):
    usage = response.get("usage", {})
    metrics = response.get("metrics", {})
    request_id = response.get("ResponseMetadata", {}).get("RequestId", "unknown")
    print(f"{label}: {message_text(response['output']['message']) or '[structured tool request]'}")
    print(f"stopReason={response.get('stopReason')} requestId={request_id}")
    print(
        "tokens="
        f"input:{usage.get('inputTokens', 0)} "
        f"output:{usage.get('outputTokens', 0)} "
        f"total:{usage.get('totalTokens', 0)}"
    )
    print(f"clientElapsedMs={elapsed * 1000:.1f} serviceLatencyMs={metrics.get('latencyMs', 'n/a')}")


def converse(messages, tool_config=None):
    request = {
        "modelId": MODEL_ID,
        "system": [{"text": SYSTEM_PROMPT}],
        "messages": messages,
        "inferenceConfig": {"temperature": 0.0, "maxTokens": 250},
    }
    if tool_config:
        request["toolConfig"] = tool_config
    started = time.perf_counter()
    response = bedrock.converse(**request)
    return response, time.perf_counter() - started


def lookup_case(case_number):
    return CASE_RECORDS.get(
        case_number,
        {"caseNumber": case_number, "error": "CASE_NOT_FOUND"},
    )


def run_message_history_demo():
    print_section("1. Converse response, usage, and explicit message history")
    first_message = {
        "role": "user",
        "content": [{"text": f"Remember that I am working on Salesforce case {CASE_NUMBER}. Acknowledge briefly."}],
    }
    first_response, elapsed = converse([first_message])
    show_response("Turn 1", first_response, elapsed)

    follow_up = {"role": "user", "content": [{"text": "Which case am I working on?"}]}
    without_history, elapsed = converse([follow_up])
    show_response("Turn 2 without history", without_history, elapsed)

    history = [first_message, first_response["output"]["message"], follow_up]
    with_history, elapsed = converse(history)
    show_response("Turn 2 with history", with_history, elapsed)
    print(f"Messages resent on the third request: {[message['role'] for message in history]}")


def run_tool_use_demo():
    print_section("2. Complete client-side tool-use exchange")
    user_message = {
        "role": "user",
        "content": [{"text": f"What are the current status and priority of Salesforce case {CASE_NUMBER}?"}],
    }
    first_response, elapsed = converse([user_message], FORCED_TOOL_CONFIG)
    show_response("Model decision", first_response, elapsed)
    if first_response.get("stopReason") != "tool_use":
        raise RuntimeError("The model did not request lookup_case; inspect the model response above.")

    assistant_message = first_response["output"]["message"]
    tool_use = next(block["toolUse"] for block in assistant_message["content"] if "toolUse" in block)
    print("Tool requested by model:")
    print(json.dumps(tool_use, indent=2))

    result = lookup_case(tool_use["input"]["caseNumber"])
    print("Result produced by application code:")
    print(json.dumps(result, indent=2))

    tool_result_message = {
        "role": "user",
        "content": [
            {
                "toolResult": {
                    "toolUseId": tool_use["toolUseId"],
                    "content": [{"json": result}],
                    "status": "success" if "error" not in result else "error",
                }
            }
        ],
    }
    final_messages = [user_message, assistant_message, tool_result_message]
    final_response, elapsed = converse(final_messages, TOOL_CONFIG)
    show_response("Grounded final answer", final_response, elapsed)
    print(f"Tool exchange roles: {[message['role'] for message in final_messages]}")


def main():
    identity = sts.get_caller_identity()
    print("Live AWS identity verified")
    print(f"region={AWS_REGION} model={MODEL_ID} principal={identity['Arn']}")
    run_message_history_demo()
    run_tool_use_demo()
    print("\nTakeaway: Converse is stateless; the application resends history, executes requested tools, and returns their results to the model.")


main()

NoCredentialsError: Unable to locate credentials